# 01 - Single Category CSMS AI Scoring Test

Evaluate a single CSMS category using the v4 knowledge base with hybrid chunking.

### Parameters
- category_id: CSMS category to test
- start_date / end_date: Evaluation period
- company: Company name or "agnostic"
- chunking_mode: memory_only | persist_selective | persist_all
- marine_dir: Path to test submissions

## 1. Setup & Configuration

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
print(f"Working directory: {os.getcwd()}")

In [ ]:
CATEGORY_ID = "8.1"
START_DATE = '2026-01-01'  # e.g. "2025-01-01"
END_DATE = '2026-10-01'    # e.g. "2025-12-31"
COMPANY = "PT. Logindo Samudramakmur Tbk."
CHUNKING_MODE = "memory_only"  # Options: memory_only, persist_selective, persist_all
PERSIST_FAILED = True
MARINE_DIR = os.path.join(os.getcwd(), "test_submissions", "marine")
JUROR_ROLE = "reasoning_model"
CONSENSUS_ROLE = "consensus_model"

print(f"Category:      {CATEGORY_ID}")
print(f"Period:        {START_DATE} to {END_DATE}")
print(f"Company:       {COMPANY}")
print(f"Chunking Mode: {CHUNKING_MODE}")
print(f"Marine Dir:    {MARINE_DIR}")

## 2. Load Knowledge Base (v4)

In [ ]:
from engine.pipeline import load_knowledge_base

kb = load_knowledge_base(CATEGORY_ID)
cat = kb['category']
vr = kb['validation_rules']
sr = kb['scoring_rules']

print(f"Category title: {cat.get('title', 'N/A')}")
print(f"KB version:     {cat.get('kb_version', 'unknown')}")
print(f"Validation methods: {vr.get('validation_methods', [])}")
print(f"Confidence threshold: {vr.get('ai_confidence_threshold', 'N/A')}")

## 3. Discover Category Folder

In [ ]:
from engine.pipeline import discover_category_folders

categories = discover_category_folders(MARINE_DIR)
target = [c for c in categories if c['category_id'] == CATEGORY_ID]

if not target:
    print(f"ERROR: Category {CATEGORY_ID} not found")
    print(f"Available: {[c['category_id'] for c in categories]}")
else:
    folder_path = target[0]['path']
    print(f"Found: {target[0]['folder_name']}")
    file_count = sum(len(files) for _, _, files in os.walk(folder_path))
    print(f"Files: {file_count}")

## 4. Run Full Pipeline Scoring

In [ ]:
from engine.pipeline import score_category
from engine.utils.eval_config import EvaluationConfig
from engine.utils.chunking_config import ChunkingMode

config = EvaluationConfig(
    start_date=START_DATE,
    end_date=END_DATE,
    company=COMPANY,
    marine_dir=MARINE_DIR,
    juror_role=JUROR_ROLE,
    consensus_role=CONSENSUS_ROLE,
    chunking_mode=ChunkingMode(CHUNKING_MODE),
    persist_failed=PERSIST_FAILED
)

print('Evaluation config:')
print(json.dumps(config.to_dict(), indent=2))
print('=" * 80')

start_time = time.time()
result = score_category(CATEGORY_ID, folder_path, config=config)
elapsed = time.time() - start_time
print(f"Total time: {elapsed:.1f}s")

## 5. Flattened Response Fields

In [ ]:
print('=" * 60')
print(f"  Category:         {result['category_id']}")
print(f"  Expected Score:   {result['expected_score']}/10")
print(f"  Compliance Score: {result['compliance_score']}/10")
print(f"  Evidence Score:   {result['evidence_score']}/10")
print(f"  HSE Score:        {result['hse_score']}/10")
print(f"  Consensus Score:  {result['consensus_score']}/10")
print(f"  Final Score:      {result['final_score']}/10")
print(f"  Require Review:   {result['require_review']}")
print(f"  Confidence:       {result['confidence']:.2f}")
print(f"  Status:           {result['status'].upper()}")
print(f"  Company:          {result['company']}")
print(f"  Chunking Mode:    {result['chunking_mode']}")
print('=" * 60')

## 6. Evidence Tracking

In [ ]:
print(f"Evidence Found ({len(result['evidence_found'])} items):")
for i, ev in enumerate(result['evidence_found'][:10], 1):
    print(f"  {i}. {ev}")

print(f"Missing Evidence ({len(result['missing_evidence'])} items):")
for i, miss in enumerate(result['missing_evidence'][:10], 1):
    print(f"  {i}. {miss}")

print(f"Errors ({len(result['errors'])}):")
for err in result['errors'][:5]:
    print(f"  - {err}")

print(f"Warnings ({len(result['warnings'])}):")
for warn in result['warnings'][:5]:
    print(f"  - {warn}")

## 7. AI Juror Reasoning

In [ ]:
for jr in result.get("reasoning_records", []):
    print(f"[{jr['juror']}] Score: {jr['recommended_score']}/10 | Conf: {jr['confidence']:.2f}")
    reasoning = jr.get("reasoning", [])
    if isinstance(reasoning, list):
        for r in reasoning:
            print(f"  - {r}")
    else:
        print(f"  {reasoning}")

print(f"--- Final Reasoning ---")
print(result['final_reasoning'])

## 8. AI Recommendations for Score 10

In [ ]:
ai_rec = result.get("ai_recommendation_for_excellent_score", {})

print('Recommendations:')
for i, rec in enumerate(ai_rec.get("recommendations", [])[:10], 1):
    print(f"  {i}. {rec}")

print('Required Documents:')
for i, doc in enumerate(ai_rec.get("required_documents", [])[:10], 1):
    print(f"  {i}. {doc}")

print('Improvement Areas:')
for i, imp in enumerate(ai_rec.get("improvement_areas", [])[:5], 1):
    print(f"  {i}. {imp}")

## 9. HTML Viewer

In [ ]:
html_path = result.get("html_viewer_path", "N/A")
print(f"HTML Viewer: {html_path}")

if os.path.exists(html_path):
    print(f"File size: {os.path.getsize(html_path)} bytes")
    print(f"Open in browser: file:///{os.path.abspath(html_path)}")

## 10. Save Full Result

In [ ]:
os.makedirs("notebook_v3/outputs", exist_ok=True)
out_path = f"notebook_v3/outputs/single_test_{CATEGORY_ID}.json"

serializable = {k: v for k, v in result.items() if not callable(v)}
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(serializable, f, indent=2, ensure_ascii=False, default=str)

print(f"Result saved to {out_path}")